# UA-DETRAC Test-Set Image Sequence to Video Converter (Kaggle Edition)

Notebook này được thiết kế chuyên biệt để **chỉ lọc và ghép các sequence thuộc tập TEST** của bộ dataset UA-DETRAC trên Kaggle. 

## 📂 Cách hoạt động:
Dataset UA-DETRAC thường chứa tất cả các thư mục ảnh trong cùng một folder lớn (không chia sẵn thư mục Train/Test riêng). Tuy nhiên, tập Test được xác định bởi sự hiện diện của các file cấu hình `.xml` trong thư mục Annotations tập Test (ví dụ: `MVI_39031.xml`).

Notebook này sẽ quét qua thư mục Annotations của tập Test, lấy ra danh sách các sequence TEST cần thiết, sau đó **chỉ tiến hành ghép video đối với các sequence có tên trong danh sách này**.

## 📂 Đường dẫn mặc định trên Kaggle:
- **Thư mục ảnh gốc (đầu vào)**: `/kaggle/input/.../DETRAC-Images/DETRAC-Images`
- **Thư mục chứa XML Annotations Test**: `/kaggle/input/.../DETRAC-Test-Annotations-XML`
- **Thư mục video kết quả (đầu ra)**: `/kaggle/working/ua_detrac_test_videos`

In [ ]:
import os
import re
import sys
from pathlib import Path

import cv2
from tqdm.auto import tqdm

print("✓ Import thành công!")
print(f"OpenCV version: {cv2.__version__}")


## 1. Định nghĩa các hàm trợ giúp

In [ ]:
def extract_frame_number(path: Path) -> int:
    """
    Trích xuất số khung hình từ tên file (ví dụ: 'img00001' -> 1).
    Giúp sắp xếp chính xác thứ tự các frame theo số học.
    """
    match = re.findall(r"\d+", path.stem)
    if match:
        return int(match[-1])
    return 0


def format_size(bytes_size: float) -> str:
    """Định dạng kích thước file dễ đọc."""
    for unit in ["B", "KB", "MB", "GB"]:
        if bytes_size < 1024.0:
            return f"{bytes_size:.2f} {unit}"
        bytes_size /= 1024.0
    return f"{bytes_size:.2f} TB"


def convert_sequence_to_video(
    sequence_dir: Path,
    output_path: Path,
    fps: int = 25,
    codec: str = "mp4v",
    img_exts: tuple = (".jpg", ".jpeg", ".png", ".bmp"),
) -> bool:
    """
    Ghép tất cả ảnh trong thư mục sequence_dir thành 1 video tại output_path.
    """
    # Tìm tất cả file ảnh phù hợp
    image_paths = [
        p
        for p in sequence_dir.iterdir()
        if p.is_file() and p.suffix.lower() in img_exts
    ]

    if not image_paths:
        print(f"  [!] Không tìm thấy ảnh nào trong sequence {sequence_dir.name}.")
        return False

    # Sắp xếp các ảnh theo số thứ tự frame
    image_paths.sort(key=extract_frame_number)

    # Đọc ảnh đầu tiên để lấy độ phân giải (Resolution)
    first_img_path = image_paths[0]
    first_frame = cv2.imread(str(first_img_path))

    if first_frame is None:
        print(f"  [ERROR] Không đọc được ảnh đầu tiên: {first_img_path}")
        return False

    height, width, _ = first_frame.shape
    print(f"  [i] Tìm thấy {len(image_paths)} ảnh. Độ phân giải: {width}x{height}")

    # Khởi tạo OpenCV VideoWriter
    fourcc = cv2.VideoWriter_fourcc(*codec)
    os.makedirs(output_path.parent, exist_ok=True)

    out_video = cv2.VideoWriter(str(output_path), fourcc, float(fps), (width, height))

    if not out_video.isOpened():
        print(f"  [ERROR] Lỗi khởi tạo VideoWriter với codec '{codec}'.")
        return False

    # Tiến hành ghi từng frame vào video
    print(f"  [>] Đang ghép video: {output_path.name}")

    # Sử dụng thanh tiến trình tqdm
    for img_path in tqdm(image_paths, desc=f"      {sequence_dir.name}", leave=False):
        frame = cv2.imread(str(img_path))
        if frame is not None:
            out_video.write(frame)
        else:
            print(f"\n  [!] Cảnh báo: Bỏ qua frame lỗi {img_path.name}")

    out_video.release()

    # Kiểm tra kích thước file kết quả
    if output_path.exists():
        file_size = output_path.stat().st_size
        print(f"  [+] Ghép thành công! Dung lượng: {format_size(file_size)}")
        return True
    else:
        print("  [ERROR] Lỗi không tạo được file video.")
        return False


## 2. Cấu hình đường dẫn trên Kaggle

Hãy chỉnh sửa các thông số dưới đây cho phù hợp với môi trường Kaggle của bạn.

In [ ]:
# === CẤU HÌNH ĐƯỜNG DẪN KAGGLE ===

# 1. Thư mục chứa các folder sequence ảnh của UA-DETRAC
IMAGES_FOLDER = (
    "/kaggle/input/datasets/bratjay/ua-detrac-orig/DETRAC-Images/DETRAC-Images"
)

# 2. Thư mục chứa XML Annotations Test (Dùng để lọc tập Test)
TEST_ANNOTATIONS_FOLDER = "/kaggle/input/datasets/bratjay/ua-detrac-orig/DETRAC-Test-Annotations-XML/DETRAC-Test-Annotations-XML"

# 3. Thư mục lưu các video kết quả sau khi ghép
OUTPUT_FOLDER = "/kaggle/working/ua_detrac_test_videos"

# 4. Tốc độ khung hình (FPS) cho video. UA-DETRAC gốc là 25 FPS.
FPS = 25

# 5. Codec video. 'mp4v' tạo file .mp4 nén rất tốt và chạy ổn định.
CODEC = "mp4v"

# --- TỰ ĐỘNG PHÁT HIỆN ĐƯỜNG DẪN TRÊN KAGGLE (ĐỀ PHÒNG ĐƯỜNG DẪN KHÁC) ---
kaggle_input = Path("/kaggle/input")

if not os.path.exists(IMAGES_FOLDER) and kaggle_input.is_dir():
    print("⚠️ Không tìm thấy IMAGES_FOLDER cấu hình sẵn, đang tự động quét...")
    matches_img = list(kaggle_input.rglob("DETRAC-Images"))
    if matches_img:
        IMAGES_FOLDER = str(matches_img[-1])
        print(f"⭐ Đã phát hiện IMAGES_FOLDER: {IMAGES_FOLDER}")

if not os.path.exists(TEST_ANNOTATIONS_FOLDER) and kaggle_input.is_dir():
    print(
        "⚠️ Không tìm thấy TEST_ANNOTATIONS_FOLDER cấu hình sẵn, đang tự động quét..."
    )
    matches_ann = list(kaggle_input.rglob("DETRAC-Test-Annotations-XML"))
    if matches_ann:
        TEST_ANNOTATIONS_FOLDER = str(matches_ann[-1])
        print(f"⭐ Đã phát hiện TEST_ANNOTATIONS_FOLDER: {TEST_ANNOTATIONS_FOLDER}")

print("-" * 80)
print(f"✓ Thư mục ảnh: {IMAGES_FOLDER}")
print(f"✓ Thư mục Annotations Test: {TEST_ANNOTATIONS_FOLDER}")
print(f"✓ Thư mục đầu ra: {OUTPUT_FOLDER}")


## 3. Thực thi quá trình lọc và ghép Video tập Test

In [ ]:
input_path = Path(IMAGES_FOLDER)
ann_path = Path(TEST_ANNOTATIONS_FOLDER)
output_path = Path(OUTPUT_FOLDER)

if not input_path.is_dir():
    raise FileNotFoundError(f"Thư mục ảnh không tồn tại: {input_path}")

# 1. Đọc danh sách các sequence TEST từ file XML trong thư mục Annotations
test_sequences = set()
if ann_path.is_dir():
    print(f"[*] Đang quét thư mục XML để lấy danh sách file test...")
    for f in ann_path.iterdir():
        if f.is_file() and f.suffix.lower() == ".xml":
            test_sequences.add(f.stem)
    print(f"✓ Tìm thấy {len(test_sequences)} sequence test trong file annotations.")
else:
    print(f"⚠️ Cảnh báo: Thư mục Annotations XML không tồn tại: {ann_path}.")
    print("Chương trình sẽ gộp toàn bộ các sequence ảnh thay vì chỉ gộp tập test.")

# 2. Quét tất cả các folder sequence ảnh đầu vào
sequence_dirs = [
    p
    for p in input_path.iterdir()
    if p.is_dir() and not p.name.startswith(".") and not p.name.startswith("_")
]

# 3. Lọc danh sách: Chỉ giữ lại các thư mục có tên trong tập TEST
if test_sequences:
    sequence_dirs = [p for p in sequence_dirs if p.name in test_sequences]
    print(f"✓ Sau khi lọc: Giữ lại {len(sequence_dirs)} sequences phù hợp tập TEST.")

if not sequence_dirs:
    print("❌ Không tìm thấy sequence nào phù hợp để xử lý!")
else:
    # Sắp xếp theo tên
    sequence_dirs.sort(key=lambda p: p.name)

    print(f"🚀 Bắt đầu ghép {len(sequence_dirs)} video tập TEST.")
    print(f"📂 Video sẽ được lưu tại: {output_path.resolve()}")
    print(f"⚡ Cấu hình: {FPS} FPS | Codec: {CODEC}")
    print("-" * 80)

    success_count = 0
    os.makedirs(output_path, exist_ok=True)

    # Vòng lặp xử lý từng sequence
    for i, seq_dir in enumerate(sequence_dirs):
        seq_name = seq_dir.name
        print(f"\n[{i+1}/{len(sequence_dirs)}] Đang xử lý sequence test: {seq_name}")

        out_video_path = output_path / f"{seq_name}.mp4"

        try:
            success = convert_sequence_to_video(
                sequence_dir=seq_dir, output_path=out_video_path, fps=FPS, codec=CODEC
            )
            if success:
                success_count += 1
        except Exception as e:
            print(f"  [ERROR] Lỗi khi xử lý sequence {seq_name}: {e}")

    print("\n" + "=" * 80)
    print(
        f"🎉 Hoàn thành: Ghép thành công {success_count} / {len(sequence_dirs)} video tập TEST."
    )
    print(f"📂 Thư mục video: {output_path.resolve()}")
    print("=" * 80)


## 4. Nén thư mục kết quả để tải về (Tùy chọn)

Cách nhanh nhất để tải xuống toàn bộ video từ Kaggle là nén chúng thành file `.zip`.

In [ ]:
import shutil

# Nén thư mục video thành file zip
zip_name = "/kaggle/working/ua_detrac_test_videos_archive"
if os.path.exists(OUTPUT_FOLDER) and len(os.listdir(OUTPUT_FOLDER)) > 0:
    print(f"📦 Đang nén thư mục {OUTPUT_FOLDER}...")
    zip_path = shutil.make_archive(zip_name, "zip", OUTPUT_FOLDER)
    print(f"🎉 Đã nén thành công! File lưu tại: {zip_path}")
    print(f"💾 Kích thước archive: {format_size(os.path.getsize(zip_path))}")
else:
    print("⚠️ Không có file video nào để nén.")
